# Experiment: Dynamic Rule Selector Demo


This notebook demos a boundary-token update loop for dynamic rule selection:

1. Load one RuleArena airline sample.
2. Get applicable rules and oracle-derived execution order.
3. Generate the answer one sentence at a time.
4. After each sentence, call an external evaluator LLM to choose which rule(s) to boost next.

For now, this notebook does selection only (no attention boosting intervention).


In [1]:
from pathlib import Path
import importlib.util
import types
import sys
import json
import os
import re
from pprint import pprint


def load_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

print("Project root:", ROOT)

seg_mod = load_module("src.rulearena.rulebook_segments", ROOT / "src/rulearena/rulebook_segments.py")
app_mod = load_module("src.rulearena.rule_applicability", ROOT / "src/rulearena/rule_applicability.py")

src_pkg = types.ModuleType("src")
src_pkg.__path__ = [str(ROOT / "src")]
sys.modules["src"] = src_pkg

rulearena_pkg = types.ModuleType("src.rulearena")
rulearena_pkg.__path__ = [str(ROOT / "src/rulearena")]
sys.modules["src.rulearena"] = rulearena_pkg

sys.modules["src.rulearena.rulebook_segments"] = seg_mod
sys.modules["src.rulearena.rule_applicability"] = app_mod

oracle_mod = load_module("src.rulearena.rule_application_oracle", ROOT / "src/rulearena/rule_application_oracle.py")

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except Exception:
    OPENAI_AVAILABLE = False


Project root: /Users/vitoriag/Documents/multi-rules


## Sample Selection


In [2]:
COMP = 0
SAMPLE_IDX = 1

rulebook_path = ROOT / "datasets/RuleArena/airline/reference_rules.txt"
problems_path = ROOT / f"datasets/RuleArena/airline/synthesized_problems/comp_{COMP}.jsonl"

with open(problems_path) as f:
    for i, line in enumerate(f):
        if i == SAMPLE_IDX:
            sample = json.loads(line)
            break
    else:
        raise IndexError(f"Sample {SAMPLE_IDX} not found in {problems_path}")

question_prompt = sample["prompt"]
info = sample["info"]

print(f"Loaded comp_{COMP} sample {SAMPLE_IDX}")
print("\nQuestion prompt:")
print(question_prompt)
print("\nInfo dict:")
pprint(info)


Loaded comp_0 sample 1

Question prompt:
Linda is a Business Class passenger flying from Charlotte to Phoenix with the following items:
1. A backpack: 18 x 13 x 6 inches, 8 lbs;
2. A luggage box: 41 x 20 x 16 inches, 95 lbs;
3. A backpack: 38 x 24 x 18 inches, 74 lbs;
4. A backpack: 37 x 16 x 10 inches, 54 lbs;
5. A backpack: 43 x 25 x 20 inches, 52 lbs;

Linda's flight ticket is $186.

Info dict:
{'bag_list': [{'id': 1, 'name': 'backpack', 'size': [18, 13, 6], 'weight': 8},
              {'id': 2,
               'name': 'luggage box',
               'size': [41, 20, 16],
               'weight': 95},
              {'id': 3, 'name': 'backpack', 'size': [38, 24, 18], 'weight': 74},
              {'id': 4, 'name': 'backpack', 'size': [37, 16, 10], 'weight': 54},
              {'id': 5,
               'name': 'backpack',
               'size': [43, 25, 20],
               'weight': 52}],
 'base_price': 186,
 'customer_class': 'Business',
 'direction': 1,
 'routine': 'U.S.'}


## Build Rule Set And Oracle Order


In [3]:
rulebook_text = rulebook_path.read_text()
coarse_segments = seg_mod.get_coarse_segments(rulebook_text)
fine_segments = seg_mod.get_fine_segments(rulebook_text)

applied = app_mod.get_applied_rules_with_coarse(info, fine_segments, coarse_segments)
trace = oracle_mod.get_rule_application_trace(info, fine_segments)

ordered_unique_rules = []
seen = set()
for step in trace["steps"]:
    name = step["rule_name"]
    if name not in seen:
        seen.add(name)
        ordered_unique_rules.append(name)

relevant_fine_rules = [seg["name"] for seg in applied["fine"]]

print("Applicable coarse sections:", len(applied["coarse"]))
print([seg["name"] for seg in applied["coarse"]])

print("\nApplicable fine rules:", len(relevant_fine_rules))
print(relevant_fine_rules)

print("\nOracle ordered unique rules:", len(ordered_unique_rules))
for i, r in enumerate(ordered_unique_rules):
    print(f"{i:2d}. {r}")


Applicable coarse sections: 9
['preamble', 'carry_on', 'checked_bags_intro', 'first_bag', 'second_bag', 'third_bag', 'fourth_bag', 'complimentary_bags', 'weight_and_size']

Applicable fine rules: 29
['preamble/all_published_bag_fees', 'carry_on/you_re_allowed_1', 'carry_on/your_personal_item_like', 'carry_on/these_don_t_count', 'carry_on/you_can_bring_only', 'carry_on/the_total_size_of', 'carry_on/your_soft_sided_garment', 'checked_bags_intro/bag_fees_have_been', 'checked_bags_intro/travel_within_between_the', 'checked_bags_intro/travel_to_from_canada', 'checked_bags_intro/all_bag_fees_are', 'first_bag/row_us_puerto_rico', 'second_bag/row_us_canada_puerto', 'third_bag/row_us_canada_puerto', 'fourth_bag/row_us_canada_puerto', 'complimentary_bags/in_some_cases_you', 'complimentary_bags/if_your_status_level', 'complimentary_bags/free_checked_bags_may', 'complimentary_bags/1st_checked_bag_is', 'complimentary_bags/or_when_traveling_to', 'complimentary_bags/1st_and_2nd_checked', 'complimenta

## Generator And Selector Setup

Configure models:
- `GEN_MODEL`: sentence-by-sentence generator
- `SELECTOR_MODEL`: evaluator that picks rules to boost next

If OpenAI is unavailable, selector falls back to a simple keyword heuristic.


In [4]:
GEN_MODEL = "gpt-oss:20b"
SELECTOR_MODEL = "gpt-oss:20b"

MAX_STEPS = 8
START_RULE_PTR = 0

USE_FILTERED_RULEBOOK = True
if USE_FILTERED_RULEBOOK:
    rules_for_prompt = app_mod.build_filtered_rulebook(info, rulebook_text, fine_segments, coarse_segments)
else:
    rules_for_prompt = rulebook_text

OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
OLLAMA_CHAT_URL = OLLAMA_BASE_URL.rstrip("/") + "/api/chat"

print("Using Ollama endpoint:", OLLAMA_CHAT_URL)
print("Generator model:", GEN_MODEL)
print("Selector model:", SELECTOR_MODEL)


Using Ollama endpoint: http://127.0.0.1:11434/api/chat
Generator model: gpt-oss:20b
Selector model: gpt-oss:20b


In [5]:
import urllib.request
import urllib.error


def ollama_chat(model: str, system: str, user: str, temperature: float = 0.0) -> str:
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
        "options": {"temperature": temperature},
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_CHAT_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            body = json.loads(resp.read().decode("utf-8"))
    except urllib.error.URLError as e:
        raise RuntimeError(f"Failed to reach Ollama at {OLLAMA_CHAT_URL}: {e}")

    return body.get("message", {}).get("content", "").strip()


def generate_next_sentence(current_generation: str) -> str:
    system = "You are a careful airline fee assistant. Continue the solution by exactly ONE sentence."
    user = (
        "You are solving an airline fee problem. Continue from the current partial solution with one next sentence only. "
        "Do not restart from scratch.\\n\\n"
        f"Problem:\\n{question_prompt}\\n\\n"
        f"Relevant Rules:\\n{rules_for_prompt}\\n\\n"
        f"Current solution so far:\\n{current_generation}\\n"
    )

    text = ollama_chat(GEN_MODEL, system, user, temperature=0.0)
    if not text:
        text = "I will now continue the fee computation."

    if not re.search(r"[.!?]$", text):
        text += "."
    return text


def heuristic_selector(current_generation: str, current_ptr: int) -> dict:
    lower = current_generation.lower()
    advance_cues = ["next", "then", "second", "third", "finally"]
    should_advance = any(cue in lower[-220:] for cue in advance_cues)

    if should_advance and current_ptr < len(ordered_unique_rules) - 1:
        new_ptr = current_ptr + 1
        selected = [ordered_unique_rules[new_ptr]]
        decision = "advance"
        conf = 0.58
    else:
        new_ptr = current_ptr
        selected = [ordered_unique_rules[current_ptr]]
        decision = "stay"
        conf = 0.45

    return {
        "decision": decision,
        "selected_rules": selected,
        "next_rule_pointer": new_ptr,
        "confidence": conf,
        "reason": "heuristic fallback",
    }


def external_selector(current_generation: str, current_ptr: int) -> dict:
    rules_context = [{"idx": i, "name": name} for i, name in enumerate(ordered_unique_rules)]

    selector_prompt = {
        "task": "Select which rule(s) should be boosted for the NEXT generation chunk.",
        "constraints": {
            "max_selected_rules": 3,
            "valid_decisions": ["stay", "advance", "jump"],
            "pointer_bounds": [0, len(ordered_unique_rules) - 1],
        },
        "current_rule_pointer": current_ptr,
        "ordered_rules": rules_context,
        "current_generation": current_generation,
        "expected_output_json": {
            "decision": "stay|advance|jump",
            "next_rule_pointer": "int",
            "selected_rule_indices": ["int"],
            "confidence": "float in [0,1]",
            "reason": "short string",
        },
    }

    system = "You are a strict JSON controller for rule selection. Output JSON only."
    user = json.dumps(selector_prompt, indent=2)

    raw = ollama_chat(SELECTOR_MODEL, system, user, temperature=0.0)
    if not raw:
        return heuristic_selector(current_generation, current_ptr)

    try:
        out = json.loads(raw)
    except Exception:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            return heuristic_selector(current_generation, current_ptr)
        out = json.loads(match.group(0))

    ptr = int(out.get("next_rule_pointer", current_ptr))
    ptr = max(0, min(ptr, len(ordered_unique_rules) - 1))

    idxs = out.get("selected_rule_indices", [])
    if not isinstance(idxs, list):
        idxs = [ptr]
    idxs = [int(i) for i in idxs if isinstance(i, int) or (isinstance(i, str) and i.isdigit())]
    idxs = [i for i in idxs if 0 <= i < len(ordered_unique_rules)]
    if len(idxs) == 0:
        idxs = [ptr]
    idxs = idxs[:3]

    confidence = out.get("confidence", 0.0)
    try:
        confidence = float(confidence)
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    return {
        "decision": str(out.get("decision", "stay")),
        "selected_rules": [ordered_unique_rules[i] for i in idxs],
        "selected_rule_indices": idxs,
        "next_rule_pointer": ptr,
        "confidence": confidence,
        "reason": str(out.get("reason", "")),
        "raw": raw,
    }


## Run Stepwise Generation And Rule Selection


In [6]:
history = []
current_generation = ""
rule_ptr = START_RULE_PTR

for step in range(1, MAX_STEPS + 1):
    sentence = generate_next_sentence(current_generation)
    current_generation = (current_generation + " " + sentence).strip()

    selection = external_selector(current_generation, rule_ptr)

    if selection["confidence"] < 0.35:
        selection = {
            "decision": "stay",
            "selected_rules": [ordered_unique_rules[rule_ptr]],
            "selected_rule_indices": [rule_ptr],
            "next_rule_pointer": rule_ptr,
            "confidence": selection["confidence"],
            "reason": "low confidence fallback",
        }

    rule_ptr = selection["next_rule_pointer"]

    row = {
        "step": step,
        "sentence": sentence,
        "decision": selection["decision"],
        "next_rule_pointer": selection["next_rule_pointer"],
        "selected_rules": selection["selected_rules"],
        "confidence": selection["confidence"],
        "reason": selection.get("reason", ""),
    }
    history.append(row)

    print(f"\n--- Step {step} ---")
    print("Sentence:", sentence)
    print(
        "Selection:",
        {
            "decision": row["decision"],
            "next_rule_pointer": row["next_rule_pointer"],
            "selected_rules": row["selected_rules"],
            "confidence": row["confidence"],
        },
    )

    if "the total cost is" in current_generation.lower():
        break



--- Step 1 ---
Sentence: Linda will pay a total of $1,400 in checked‑bag fees, bringing her overall travel cost to $1,586 when adding her $186 ticket.
Selection: {'decision': 'jump', 'next_rule_pointer': 7, 'selected_rules': ['checked_bags_intro/bag_fees_have_been', 'checked_bags_intro/all_bag_fees_are'], 'confidence': 0.85}



--- Step 2 ---
Sentence: The $1,400 figure comes from adding the $40 first‑bag fee, $45 second‑bag fee, $150 third‑bag fee, and $200 each for the fourth and fifth bags, plus the $200 overweight fee for the 95‑lb bag, the $200 overweight fee for the 74‑lb bag, and the $100 overweight fee for the 54‑lb bag.
Selection: {'decision': 'advance', 'next_rule_pointer': 8, 'selected_rules': ['checked_bags_intro/travel_within_between_the'], 'confidence': 0.9}



--- Step 3 ---
Sentence: However, the correct total for Linda’s checked‑bag fees is $1,135, not $1,400, because the overweight and oversize fees for the 95‑lb, 74‑lb, 54‑lb, and 52‑lb bags are $200, $200, $100, and $200 respectively, plus the standard bag fees of $40, $45, $150, and $200.
Selection: {'decision': 'jump', 'next_rule_pointer': 10, 'selected_rules': ['checked_bags_intro/travel_within_between_the', 'checked_bags_intro/travel_to_from_canada', 'checked_bags_intro/all_bag_fees_are'], 'confidence': 0.85}



--- Step 4 ---
Sentence: Adding those amounts gives a total of $1,135 in checked‑bag fees, so Linda’s overall travel cost is $1,321 ($186 ticket plus $1,135 fees).
Selection: {'decision': 'stay', 'next_rule_pointer': 10, 'selected_rules': ['checked_bags_intro/all_bag_fees_are'], 'confidence': 0.92}



--- Step 5 ---
Sentence: Thus, Linda should budget $1,321 for her trip, assuming no complimentary bag status applies.
Selection: {'decision': 'advance', 'next_rule_pointer': 11, 'selected_rules': ['complimentary_bags/in_some_cases_you', 'complimentary_bags/if_your_status_level'], 'confidence': 0.8}



--- Step 6 ---
Sentence: If Linda holds AAdvantage Platinum status or a oneworld Sapphire membership, she would receive the first two bags free, reducing her total cost to $1,186.
Selection: {'decision': 'advance', 'next_rule_pointer': 12, 'selected_rules': ['complimentary_bags/if_your_status_level', 'complimentary_bags/free_checked_bags_may'], 'confidence': 0.92}



--- Step 7 ---
Sentence: Alternatively, Linda could repackage her items to meet the 62‑inch size limit and avoid the $200 overweight fee on the 95‑lb bag, potentially saving $200 on her total fees.
Selection: {'decision': 'advance', 'next_rule_pointer': 13, 'selected_rules': ['complimentary_bags/if_your_status_level', 'complimentary_bags/free_checked_bags_may', 'complimentary_bags/1st_checked_bag_is'], 'confidence': 0.9}



--- Step 8 ---
Sentence: If she opts to remove the 95‑lb bag entirely or replace it with a lighter, compliant bag, she would avoid the $200 overweight charge and reduce her total fees to $935, bringing her overall cost to $1,121.
Selection: {'decision': 'stay', 'next_rule_pointer': 13, 'selected_rules': ['complimentary_bags/free_checked_bags_may'], 'confidence': 0.92}


## Inspect Outputs


In [7]:
print("Final generation:\n")
print(current_generation)

print("\n\nSelection history:\n")
for row in history:
    print(json.dumps(row, indent=2))


Final generation:

Linda will pay a total of $1,400 in checked‑bag fees, bringing her overall travel cost to $1,586 when adding her $186 ticket. The $1,400 figure comes from adding the $40 first‑bag fee, $45 second‑bag fee, $150 third‑bag fee, and $200 each for the fourth and fifth bags, plus the $200 overweight fee for the 95‑lb bag, the $200 overweight fee for the 74‑lb bag, and the $100 overweight fee for the 54‑lb bag. However, the correct total for Linda’s checked‑bag fees is $1,135, not $1,400, because the overweight and oversize fees for the 95‑lb, 74‑lb, 54‑lb, and 52‑lb bags are $200, $200, $100, and $200 respectively, plus the standard bag fees of $40, $45, $150, and $200. Adding those amounts gives a total of $1,135 in checked‑bag fees, so Linda’s overall travel cost is $1,321 ($186 ticket plus $1,135 fees). Thus, Linda should budget $1,321 for her trip, assuming no complimentary bag status applies. If Linda holds AAdvantage Platinum status or a oneworld Sapphire membership,

## Notes

- This demo updates selector decisions at sentence boundaries.
- No boosting is applied yet; this is only to validate selector behavior.
- Next step: wire `selected_rules` into your attention-bias mask update at each boundary.
